# AFD calcium imaging tracker: green-channel-only with candidate ROI selector, manual click ROIs, ROI center track QC, and inline tracking movie

This version avoids red-green matching and avoids needing hover coordinates.

ROI selection has two parts:

1. **Automatic candidate ROIs:** the notebook detects bright candidate AFD soma-like spots and labels them with candidate IDs.
2. **Optional manual click ROIs:** after selecting candidate IDs, you can open an external Tk/macOS matplotlib window and click any missing AFDs.

QC tools included:

- **ROI center track QC** plots the center of each ROI across all frames.
- **Inline tracking movie** renders the tracked ROIs on the green-channel image and displays a sampled movie inside the notebook, for example every 50 frames.

The ROI size is controlled by `ROI_RADIUS` in Cell 2. Smaller values give smaller extraction ROIs.

In [ ]:
# ============================================================
# CELL 1: INSTALL PACKAGES, IMPORT PACKAGES, SET INLINE PLOTTING
# ============================================================

import sys
import subprocess
import importlib.util

required_packages = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "tifffile": "tifffile",
    "skimage": "scikit-image",
    "tqdm": "tqdm",
}

missing_packages = []
for import_name, pip_name in required_packages.items():
    if importlib.util.find_spec(import_name) is None:
        missing_packages.append(pip_name)

if len(missing_packages) > 0:
    print("Installing missing packages:")
    print(missing_packages)
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing_packages])
    print("Package installation finished.")
    print("IMPORTANT: restart the kernel after this cell if anything was newly installed.")
else:
    print("All required packages are already installed.")

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

from tifffile import TiffFile
from skimage.feature import match_template, peak_local_max
from skimage.draw import disk
from skimage.filters import gaussian
from tqdm.notebook import tqdm

warnings.filterwarnings("ignore", category=RuntimeWarning)

plt.rcParams["figure.dpi"] = 130
plt.rcParams["savefig.dpi"] = 200

print("Current matplotlib backend:", matplotlib.get_backend())

In [ ]:
# ============================================================
# CELL 2: USER PARAMETERS
# ============================================================

TIFF_PATH = "/Users/sk3526/Desktop/calcium_imaging_data/20260513_AFD_mcherry_test_baseline_test_rep_2_MMStack_Pos0.ome.tif"

OUTPUT_DIR = "/Users/sk3526/Desktop/calcium_imaging_data/AFD_green_tracking_output_rep2_candidate_selector"
os.makedirs(OUTPUT_DIR, exist_ok=True)

FPS = 4.0
N_FRAMES_TO_ANALYZE = None

FULL_FRAME_WIDTH = 512
SPLIT_X = FULL_FRAME_WIDTH // 2

# Use a bright frame for ROI selection. Your screenshot suggested 1475 is a good frame.
ROI_SELECTION_FRAME_IMAGEJ = 1475
ROI_SELECTION_FRAME = ROI_SELECTION_FRAME_IMAGEJ - 1

REFERENCE_FRAME_IMAGEJ = 574
REFERENCE_FRAME = REFERENCE_FRAME_IMAGEJ - 1

STIM_FRAMES_IMAGEJ = [1475]
STIM_FRAMES = [f - 1 for f in STIM_FRAMES_IMAGEJ]

START_FRAME_IMAGEJ = 1
END_FRAME_IMAGEJ = None
START_FRAME = START_FRAME_IMAGEJ - 1

# ROI_RADIUS is the radius in pixels.
# Approximate ROI diameter = 2 * ROI_RADIUS + 1 pixels.
# Previous version used ROI_RADIUS = 5, approximately 11-pixel diameter.
# Try 3 or 4 for smaller soma ROIs.
ROI_RADIUS = 3
BACKGROUND_INNER_RADIUS = 6
BACKGROUND_OUTER_RADIUS = 12

TEMPLATE_RADIUS = 10
SEARCH_RADIUS = 18
GAUSSIAN_SIGMA_FOR_TRACKING = 1.0

MIN_CORRELATION_ALLOWED = 0.35
MAX_DISPLACEMENT_ALLOWED = 15
HOLD_POSITION_ON_BAD_TRACK = True
UPDATE_TEMPLATE_IF_GOOD = False

# Candidate detector settings.
CANDIDATE_GAUSSIAN_SIGMA = 1.0
CANDIDATE_MIN_DISTANCE = 8
CANDIDATE_THRESHOLD_PERCENTILE = 99.2
MAX_CANDIDATES = 60
EXCLUDE_BORDER_PIXELS = 6

# Static grid settings for coordinate reading.
MAJOR_TICK_STEP = 50
MINOR_TICK_STEP = 10

print("ROI selection ImageJ frame:", ROI_SELECTION_FRAME_IMAGEJ)
print("Stim ImageJ frames:", STIM_FRAMES_IMAGEJ)

In [ ]:
# ============================================================
# CELL 3: LOAD TIFF LAZILY
# ============================================================

tif = TiffFile(TIFF_PATH)
series = tif.series[0]
movie = series.asarray(out="memmap")
movie = np.squeeze(movie)

print("Movie shape:", movie.shape)
print("Movie dtype:", movie.dtype)

if movie.ndim != 3:
    raise ValueError(f"Expected movie shape [frames, height, width], but got {movie.shape}")

n_total_frames, frame_h, frame_w = movie.shape

if N_FRAMES_TO_ANALYZE is None:
    n_frames = n_total_frames
else:
    n_frames = min(N_FRAMES_TO_ANALYZE, n_total_frames)

if END_FRAME_IMAGEJ is None:
    END_FRAME = n_frames - 1
else:
    END_FRAME = min(END_FRAME_IMAGEJ - 1, n_frames - 1)

print("Total frames:", n_total_frames)
print("Frames to analyze:", n_frames)
print("Frame size:", frame_h, "x", frame_w)
print("Analysis ImageJ frames:", START_FRAME + 1, "to", END_FRAME + 1)

if frame_w != FULL_FRAME_WIDTH:
    print("Warning: frame width does not match FULL_FRAME_WIDTH.")

In [ ]:
# ============================================================
# CELL 4: BASIC IMAGE AND DISPLAY FUNCTIONS
# ============================================================

def get_frame(frame_idx):
    return movie[frame_idx].astype(np.float32)

def split_channels(frame):
    green = frame[:, :SPLIT_X]
    red = frame[:, SPLIT_X:]
    return green, red

def get_green_frame(frame_idx):
    frame = get_frame(frame_idx)
    green, _ = split_channels(frame)
    return green

def normalize_for_display(img, p_low=1, p_high=99.7):
    lo, hi = np.percentile(img, [p_low, p_high])
    if hi <= lo:
        return img
    out = (img - lo) / (hi - lo)
    return np.clip(out, 0, 1)

def add_coordinate_grid(ax, img_shape):
    h, w = img_shape
    ax.set_xlim(0, w - 1)
    ax.set_ylim(h - 1, 0)
    ax.set_xticks(np.arange(0, w, MAJOR_TICK_STEP))
    ax.set_yticks(np.arange(0, h, MAJOR_TICK_STEP))
    ax.set_xticks(np.arange(0, w, MINOR_TICK_STEP), minor=True)
    ax.set_yticks(np.arange(0, h, MINOR_TICK_STEP), minor=True)
    ax.grid(which="major", linewidth=0.7, alpha=0.45)
    ax.grid(which="minor", linewidth=0.3, alpha=0.20)
    ax.set_xlabel("x coordinate in green channel")
    ax.set_ylabel("y coordinate")

def show_green_frame_big(frame_idx, title_extra="", figsize=(9, 10)):
    green = get_green_frame(frame_idx)
    fig, ax = plt.subplots(figsize=figsize)
    ax.imshow(normalize_for_display(green), cmap="gray")
    title = f"Green / GCaMP, Python frame {frame_idx}, ImageJ frame {frame_idx + 1}"
    if title_extra:
        title += "\n" + title_extra
    ax.set_title(title)
    add_coordinate_grid(ax, green.shape)
    plt.tight_layout()
    plt.show()

def show_green_and_red(frame_idx):
    frame = get_frame(frame_idx)
    green, red = split_channels(frame)
    fig, axes = plt.subplots(1, 2, figsize=(11, 5))
    axes[0].imshow(normalize_for_display(green), cmap="gray")
    axes[0].set_title(f"Green / GCaMP\nImageJ frame {frame_idx + 1}")
    axes[1].imshow(normalize_for_display(red), cmap="gray")
    axes[1].set_title(f"Red / mCherry\nImageJ frame {frame_idx + 1}")
    for ax, img in zip(axes, [green, red]):
        add_coordinate_grid(ax, img.shape)
    plt.tight_layout()
    plt.show()

def parse_id_list(s):
    s = s.strip().replace(" ", "")
    if len(s) == 0:
        return []
    return [int(x) for x in s.split(",") if len(x) > 0]

def parse_xy_string(s):
    parts = s.replace(" ", "").split(",")
    if len(parts) != 2:
        raise ValueError("Please enter coordinates as x,y")
    return float(parts[0]), float(parts[1])

def parse_many_xy_string(s):
    points = []
    for chunk in s.split(";"):
        chunk = chunk.strip()
        if len(chunk) > 0:
            points.append(parse_xy_string(chunk))
    return points

def validate_green_coordinate(x, y):
    if not (0 <= x < SPLIT_X):
        print(f"Warning: x={x:.2f} is outside green-channel range 0 to {SPLIT_X - 1}.")
    if not (0 <= y < frame_h):
        print(f"Warning: y={y:.2f} is outside image height 0 to {frame_h - 1}.")

show_green_and_red(REFERENCE_FRAME)
show_green_frame_big(ROI_SELECTION_FRAME, title_extra="Frame used for candidate ROI selection")

In [ ]:
# ============================================================
# CELL 5: DETECT AND DISPLAY CANDIDATE GREEN ROIS
# ============================================================

def circular_mask_values(img, x, y, radius):
    rr, cc = disk((y, x), radius, shape=img.shape)
    return img[rr, cc]

def annulus_background_values(img, x, y, inner_radius, outer_radius):
    rr_outer, cc_outer = disk((y, x), outer_radius, shape=img.shape)
    rr_inner, cc_inner = disk((y, x), inner_radius, shape=img.shape)
    mask_outer = np.zeros(img.shape, dtype=bool)
    mask_inner = np.zeros(img.shape, dtype=bool)
    mask_outer[rr_outer, cc_outer] = True
    mask_inner[rr_inner, cc_inner] = True
    annulus = mask_outer & (~mask_inner)
    return img[annulus]

def candidate_score(green_img, x, y):
    roi_vals = circular_mask_values(green_img, x, y, ROI_RADIUS)
    bg_vals = annulus_background_values(green_img, x, y, BACKGROUND_INNER_RADIUS, BACKGROUND_OUTER_RADIUS)
    roi_mean = float(np.nanmean(roi_vals)) if len(roi_vals) else np.nan
    bg_median = float(np.nanmedian(bg_vals)) if len(bg_vals) else np.nan
    score = roi_mean - bg_median
    return roi_mean, bg_median, score

def detect_candidate_rois_green(frame_idx):
    green = get_green_frame(frame_idx)
    smooth = gaussian(green, sigma=CANDIDATE_GAUSSIAN_SIGMA, preserve_range=True)
    threshold_abs = np.percentile(smooth, CANDIDATE_THRESHOLD_PERCENTILE)

    coords_yx = peak_local_max(
        smooth,
        min_distance=CANDIDATE_MIN_DISTANCE,
        threshold_abs=threshold_abs,
        exclude_border=EXCLUDE_BORDER_PIXELS
    )

    rows = []
    for y, x in coords_yx:
        roi_mean, bg_median, score = candidate_score(green, x, y)
        rows.append({
            "candidate_id": len(rows) + 1,
            "x_green": float(x),
            "y_green": float(y),
            "smoothed_intensity": float(smooth[y, x]),
            "roi_mean": roi_mean,
            "bg_median": bg_median,
            "score_roi_minus_bg": score
        })

    candidates_df = pd.DataFrame(rows)

    if len(candidates_df) == 0:
        print("No candidates detected. Try lowering CANDIDATE_THRESHOLD_PERCENTILE.")
        return candidates_df

    candidates_df = (
        candidates_df
        .sort_values("score_roi_minus_bg", ascending=False)
        .head(MAX_CANDIDATES)
        .reset_index(drop=True)
    )

    candidates_df["candidate_id"] = np.arange(1, len(candidates_df) + 1)
    return candidates_df

def plot_candidate_rois(frame_idx, candidates_df):
    green = get_green_frame(frame_idx)
    fig, ax = plt.subplots(figsize=(10, 12))
    ax.imshow(normalize_for_display(green), cmap="gray")
    ax.set_title(f"Candidate green-channel ROIs\nImageJ frame {frame_idx + 1}\nType candidate IDs in the next cell")
    add_coordinate_grid(ax, green.shape)

    for _, row in candidates_df.iterrows():
        cid = int(row["candidate_id"])
        x = float(row["x_green"])
        y = float(row["y_green"])
        circle = plt.Circle((x, y), ROI_RADIUS, fill=False, linewidth=1.2)
        ax.add_patch(circle)
        ax.text(
            x + ROI_RADIUS + 1,
            y,
            str(cid),
            fontsize=8,
            bbox=dict(facecolor="white", alpha=0.65, edgecolor="none", pad=1)
        )

    plt.tight_layout()
    plt.show()

candidates_df = detect_candidate_rois_green(ROI_SELECTION_FRAME)

candidate_path = os.path.join(OUTPUT_DIR, "green_roi_candidates.csv")
candidates_df.to_csv(candidate_path, index=False)

print("Candidate ROI table:")
display(candidates_df[["candidate_id", "x_green", "y_green", "score_roi_minus_bg", "roi_mean", "bg_median"]])
print("Saved:", candidate_path)

plot_candidate_rois(ROI_SELECTION_FRAME, candidates_df)

In [ ]:
# ============================================================
# CELL 6: SELECT ROIS BY CANDIDATE ID
# ============================================================
#
# Enter candidate IDs, for example:
#     1,4,7,11
#
# Optional manual additions can be entered as:
#     112,245; 118,263
#
# You can leave the manual entry blank.
# ============================================================

def select_rois_from_candidates(frame_idx, candidates_df):
    selected_id_text = input("Enter candidate IDs to keep, separated by commas: ")
    manual_text = input("Optional: enter extra manual GREEN coordinates as x,y; x,y, or leave blank: ")

    selected_ids = parse_id_list(selected_id_text)

    roi_rows = []

    for cid in selected_ids:
        match = candidates_df[candidates_df["candidate_id"] == cid]
        if len(match) == 0:
            print(f"Warning: candidate ID {cid} was not found. Skipping.")
            continue

        row = match.iloc[0]
        x = float(row["x_green"])
        y = float(row["y_green"])
        validate_green_coordinate(x, y)

        roi_rows.append({
            "roi_id": len(roi_rows) + 1,
            "x_green_initial": x,
            "y_green_initial": y,
            "frame_initial_python": int(frame_idx),
            "frame_initial_imagej": int(frame_idx + 1),
            "roi_radius": ROI_RADIUS,
            "source": f"candidate_{cid}"
        })

    manual_text = manual_text.strip()
    if len(manual_text) > 0:
        manual_points = parse_many_xy_string(manual_text)
        for x, y in manual_points:
            validate_green_coordinate(x, y)
            roi_rows.append({
                "roi_id": len(roi_rows) + 1,
                "x_green_initial": float(x),
                "y_green_initial": float(y),
                "frame_initial_python": int(frame_idx),
                "frame_initial_imagej": int(frame_idx + 1),
                "roi_radius": ROI_RADIUS,
                "source": "manual_coordinate"
            })

    if len(roi_rows) == 0:
        raise RuntimeError("No ROIs were selected.")

    roi_df = pd.DataFrame(roi_rows)

    roi_path = os.path.join(OUTPUT_DIR, "selected_green_rois.csv")
    roi_df.to_csv(roi_path, index=False)

    print("Selected ROIs:")
    display(roi_df)
    print("Saved:", roi_path)

    return roi_df

roi_df = select_rois_from_candidates(ROI_SELECTION_FRAME, candidates_df)

In [ ]:
# ============================================================
# CELL 6B: OPTIONAL - ADD MISSING ROIS BY CLICKING
# ============================================================
#
# Use this cell only if automatic candidate detection missed some AFDs.
#
# How it works:
#   1. This cell opens an external matplotlib window.
#   2. LEFT-CLICK missing AFD soma centers.
#   3. Press ENTER/RETURN when done.
#   4. The clicked ROIs are appended to roi_df.
#
# On macOS:
#   - Try MANUAL_CLICK_BACKEND = "tk" first.
#   - If the external window is blank, try MANUAL_CLICK_BACKEND = "osx".
#
# If you do not need extra ROIs, set ADD_EXTRA_ROIS_BY_CLICK = False.
# ============================================================

import matplotlib.patches as patches

ADD_EXTRA_ROIS_BY_CLICK = True

# Try "tk" first.
# If the window is blank on macOS, change this to "osx" and rerun this cell.
MANUAL_CLICK_BACKEND = "tk"

EXTRA_ROI_FRAME_IMAGEJ = ROI_SELECTION_FRAME_IMAGEJ
EXTRA_ROI_FRAME = EXTRA_ROI_FRAME_IMAGEJ - 1


def plot_rois_on_green_local(
    frame_idx,
    roi_positions_df,
    x_col="x_green",
    y_col="y_green",
    title="ROIs on green channel"
):
    """
    Local plotting function for Cell 6B.
    This avoids NameError if the main plot_rois_on_green() function
    has not been defined yet.
    """
    green = get_green_frame(frame_idx)

    fig, ax = plt.subplots(figsize=(9, 10))
    ax.imshow(normalize_for_display(green), cmap="gray")
    ax.set_title(f"{title}\\nImageJ frame {frame_idx + 1}")

    # Use coordinate grid if available.
    try:
        add_coordinate_grid(ax, green.shape)
    except NameError:
        ax.set_xlabel("x coordinate in green channel")
        ax.set_ylabel("y coordinate")

    for _, row in roi_positions_df.iterrows():
        roi_id = int(row["roi_id"])
        x = float(row[x_col])
        y = float(row[y_col])

        circle = plt.Circle(
            (x, y),
            ROI_RADIUS,
            fill=False,
            linewidth=1.5
        )
        ax.add_patch(circle)
        ax.plot(x, y, "o", markersize=3)
        ax.text(
            x + ROI_RADIUS + 2,
            y,
            str(roi_id),
            fontsize=10,
            bbox=dict(
                facecolor="white",
                alpha=0.65,
                edgecolor="none",
                pad=1
            )
        )

    plt.tight_layout()
    plt.show()


def append_manual_click_rois(roi_df, frame_idx, backend="tk"):
    """
    Open an external matplotlib window and append clicked ROIs to roi_df.
    """
    green = get_green_frame(frame_idx)
    vmin, vmax = np.percentile(green, [1, 99.7])

    clicked_points = []

    try:
        print(f"Switching matplotlib backend to: {backend}")
        get_ipython().run_line_magic("matplotlib", backend)

        fig, ax = plt.subplots(figsize=(9, 9))
        ax.imshow(green, cmap="gray", vmin=vmin, vmax=vmax)
        ax.set_title(
            "LEFT-CLICK missing AFD soma centers\\n"
            "Press ENTER/RETURN when done",
            fontsize=11
        )
        ax.set_xlabel("x coordinate in green channel")
        ax.set_ylabel("y coordinate")

        # Show already selected ROIs.
        for _, row in roi_df.iterrows():
            roi_id = int(row["roi_id"])
            x = float(row["x_green_initial"])
            y = float(row["y_green_initial"])

            rect = patches.Rectangle(
                (x - ROI_RADIUS, y - ROI_RADIUS),
                2 * ROI_RADIUS + 1,
                2 * ROI_RADIUS + 1,
                linewidth=1.5,
                edgecolor="yellow",
                facecolor="none"
            )
            ax.add_patch(rect)

            ax.text(
                x + ROI_RADIUS + 3,
                y,
                f"ROI {roi_id}",
                color="yellow",
                fontsize=9,
                bbox=dict(
                    facecolor="black",
                    alpha=0.5,
                    edgecolor="none",
                    pad=1
                )
            )

        plt.tight_layout()
        plt.show()

        clicked_points = plt.ginput(
            n=-1,
            timeout=0,
            show_clicks=True
        )

        plt.close(fig)

    except Exception as err:
        print("Manual clicking did not work in this environment.")
        print("Error:", err)
        clicked_points = []

    finally:
        try:
            get_ipython().run_line_magic("matplotlib", "inline")
            print("Switched matplotlib backend back to inline.")
        except Exception:
            pass

    if len(clicked_points) == 0:
        print("No extra clicked ROIs were added.")
        return roi_df.copy()

    new_rows = []
    start_id = int(roi_df["roi_id"].max()) + 1

    for i, (x, y) in enumerate(clicked_points):
        x = int(round(x))
        y = int(round(y))

        validate_green_coordinate(x, y)

        new_rows.append({
            "roi_id": start_id + i,
            "x_green_initial": float(x),
            "y_green_initial": float(y),
            "frame_initial_python": int(frame_idx),
            "frame_initial_imagej": int(frame_idx + 1),
            "roi_radius": ROI_RADIUS,
            "source": "manual_click"
        })

    extra_df = pd.DataFrame(new_rows)
    updated_roi_df = pd.concat([roi_df, extra_df], ignore_index=True)

    roi_path = os.path.join(
        OUTPUT_DIR,
        "selected_green_rois_with_manual_clicks.csv"
    )
    updated_roi_df.to_csv(roi_path, index=False)

    print(f"Added {len(extra_df)} extra ROI(s).")
    print("Updated ROI table:")
    display(updated_roi_df)
    print("Saved:", roi_path)

    return updated_roi_df


if ADD_EXTRA_ROIS_BY_CLICK:
    roi_df = append_manual_click_rois(
        roi_df,
        frame_idx=EXTRA_ROI_FRAME,
        backend=MANUAL_CLICK_BACKEND
    )

    # Confirmation plot.
    initial_plot_df = roi_df.rename(columns={
        "x_green_initial": "x_green",
        "y_green_initial": "y_green"
    })

    plot_rois_on_green_local(
        EXTRA_ROI_FRAME,
        initial_plot_df,
        x_col="x_green",
        y_col="y_green",
        title="Selected ROIs after optional manual clicking"
    )

else:
    print("Skipping manual click ROI addition.")

In [ ]:
# ============================================================
# CELL 7: VISUALIZE SELECTED ROIS
# ============================================================

def plot_rois_on_green(frame_idx, roi_positions_df, x_col="x_green", y_col="y_green", title="ROIs on green channel"):
    green = get_green_frame(frame_idx)
    fig, ax = plt.subplots(figsize=(9, 10))
    ax.imshow(normalize_for_display(green), cmap="gray")
    ax.set_title(f"{title}\nImageJ frame {frame_idx + 1}")
    add_coordinate_grid(ax, green.shape)

    for _, row in roi_positions_df.iterrows():
        roi_id = int(row["roi_id"])
        x = float(row[x_col])
        y = float(row[y_col])
        circle = plt.Circle((x, y), ROI_RADIUS, fill=False, linewidth=1.5)
        ax.add_patch(circle)
        ax.plot(x, y, "o", markersize=3)
        ax.text(
            x + ROI_RADIUS + 2,
            y,
            str(roi_id),
            fontsize=10,
            bbox=dict(facecolor="white", alpha=0.65, edgecolor="none", pad=1)
        )

    plt.tight_layout()
    plt.show()

initial_plot_df = roi_df.rename(columns={"x_green_initial": "x_green", "y_green_initial": "y_green"})

plot_rois_on_green(
    ROI_SELECTION_FRAME,
    initial_plot_df,
    x_col="x_green",
    y_col="y_green",
    title="Selected green-channel ROIs"
)

In [ ]:
# ============================================================
# CELL 8: TRACKING HELPER FUNCTIONS
# ============================================================

def crop_with_bounds(img, x, y, radius):
    h, w = img.shape
    x = int(round(x))
    y = int(round(y))
    x0 = max(0, x - radius)
    x1 = min(w, x + radius + 1)
    y0 = max(0, y - radius)
    y1 = min(h, y + radius + 1)
    crop = img[y0:y1, x0:x1]
    return crop, x0, x1, y0, y1

def make_template(green_img, x, y, template_radius):
    smooth = gaussian(green_img, sigma=GAUSSIAN_SIGMA_FOR_TRACKING, preserve_range=True)
    template, x0, x1, y0, y1 = crop_with_bounds(smooth, x, y, template_radius)
    return template

def track_one_step_green(green_img, prev_x, prev_y, template):
    smooth = gaussian(green_img, sigma=GAUSSIAN_SIGMA_FOR_TRACKING, preserve_range=True)
    search_crop, sx0, sx1, sy0, sy1 = crop_with_bounds(smooth, prev_x, prev_y, SEARCH_RADIUS)

    if search_crop.shape[0] < template.shape[0] or search_crop.shape[1] < template.shape[1]:
        return float(prev_x), float(prev_y), np.nan, False, 0.0

    result = match_template(search_crop, template, pad_input=False)
    y_peak, x_peak = np.unravel_index(np.argmax(result), result.shape)
    corr = float(result[y_peak, x_peak])

    template_h, template_w = template.shape
    new_x = sx0 + x_peak + (template_w - 1) / 2
    new_y = sy0 + y_peak + (template_h - 1) / 2

    displacement = float(np.sqrt((new_x - prev_x) ** 2 + (new_y - prev_y) ** 2))

    is_good = True
    if not np.isfinite(corr):
        is_good = False
    if corr < MIN_CORRELATION_ALLOWED:
        is_good = False
    if displacement > MAX_DISPLACEMENT_ALLOWED:
        is_good = False

    return float(new_x), float(new_y), corr, is_good, displacement

def extract_green_intensity(green_img, x, y):
    roi_vals = circular_mask_values(green_img, x, y, ROI_RADIUS)
    bg_vals = annulus_background_values(green_img, x, y, BACKGROUND_INNER_RADIUS, BACKGROUND_OUTER_RADIUS)

    roi_mean = float(np.nanmean(roi_vals)) if len(roi_vals) > 0 else np.nan
    roi_median = float(np.nanmedian(roi_vals)) if len(roi_vals) > 0 else np.nan
    bg_mean = float(np.nanmean(bg_vals)) if len(bg_vals) > 0 else np.nan
    bg_median = float(np.nanmedian(bg_vals)) if len(bg_vals) > 0 else np.nan

    corrected_mean = roi_mean - bg_median

    return roi_mean, roi_median, bg_mean, bg_median, corrected_mean

In [ ]:
# ============================================================
# CELL 9: KEYFRAME SETUP
# ============================================================
#
# Optional correction keyframes:
# If ROI 3 fails at ImageJ frame 1200, add:
# {"roi_id": 3, "frame_imagej": 1200, "x_green": 123, "y_green": 250}
# Then rerun Cell 9 onward.
# ============================================================

MANUAL_KEYFRAMES = [
    # {"roi_id": 1, "frame_imagej": 1200, "x_green": 123, "y_green": 250},
]

def build_keyframes_df(roi_df, manual_keyframes):
    rows = []

    for _, row in roi_df.iterrows():
        rows.append({
            "roi_id": int(row["roi_id"]),
            "frame": int(row["frame_initial_python"]),
            "frame_imagej": int(row["frame_initial_imagej"]),
            "x_green": float(row["x_green_initial"]),
            "y_green": float(row["y_green_initial"]),
            "source": "initial"
        })

    for item in manual_keyframes:
        frame_imagej = int(item["frame_imagej"])
        frame_python = frame_imagej - 1
        rows.append({
            "roi_id": int(item["roi_id"]),
            "frame": int(frame_python),
            "frame_imagej": int(frame_imagej),
            "x_green": float(item["x_green"]),
            "y_green": float(item["y_green"]),
            "source": "manual_correction"
        })

    keyframes_df = pd.DataFrame(rows)
    keyframes_df = keyframes_df.sort_values(["roi_id", "frame"]).reset_index(drop=True)

    for _, row in keyframes_df.iterrows():
        validate_green_coordinate(row["x_green"], row["y_green"])

    keyframes_path = os.path.join(OUTPUT_DIR, "green_tracking_keyframes.csv")
    keyframes_df.to_csv(keyframes_path, index=False)

    print("Keyframes:")
    display(keyframes_df)
    print("Saved:", keyframes_path)

    return keyframes_df

keyframes_df = build_keyframes_df(roi_df, MANUAL_KEYFRAMES)

In [ ]:
# ============================================================
# CELL 10: TRACKING FUNCTIONS WITH KEYFRAME SUPPORT
# ============================================================

def track_roi_segment(roi_id, start_frame, end_frame, start_x, start_y, direction):
    if direction not in [+1, -1]:
        raise ValueError("direction must be +1 or -1")

    if direction == +1 and end_frame < start_frame:
        return pd.DataFrame()

    if direction == -1 and end_frame > start_frame:
        return pd.DataFrame()

    green_start = get_green_frame(start_frame)
    template = make_template(green_start, start_x, start_y, TEMPLATE_RADIUS)

    current_x = float(start_x)
    current_y = float(start_y)
    rows = []
    frame_range = range(start_frame, end_frame + direction, direction)

    for frame_idx in frame_range:
        green = get_green_frame(frame_idx)

        if frame_idx == start_frame:
            corr = 1.0
            is_good = True
            displacement = 0.0
        else:
            x_new, y_new, corr, is_good, displacement = track_one_step_green(green, current_x, current_y, template)

            if (not is_good) and HOLD_POSITION_ON_BAD_TRACK:
                x_pred = current_x
                y_pred = current_y
            else:
                x_pred = x_new
                y_pred = y_new

            current_x = float(x_pred)
            current_y = float(y_pred)

            if UPDATE_TEMPLATE_IF_GOOD and is_good:
                template = make_template(green, current_x, current_y, TEMPLATE_RADIUS)

        green_roi_mean, green_roi_median, green_bg_mean, green_bg_median, green_corrected = extract_green_intensity(
            green, current_x, current_y
        )

        rows.append({
            "frame": int(frame_idx),
            "frame_imagej": int(frame_idx + 1),
            "time_sec": float(frame_idx / FPS),
            "roi_id": int(roi_id),
            "x_green": float(current_x),
            "y_green": float(current_y),
            "green_raw_mean": green_roi_mean,
            "green_raw_median": green_roi_median,
            "green_bg_mean": green_bg_mean,
            "green_bg_median": green_bg_median,
            "green_corrected": green_corrected,
            "tracking_corr": corr,
            "tracking_good": bool(is_good),
            "tracking_displacement": displacement,
            "tracking_direction": "forward" if direction == +1 else "backward",
            "segment_start_frame": int(start_frame),
            "segment_start_imagej": int(start_frame + 1)
        })

    return pd.DataFrame(rows)

def track_single_roi_with_keyframes(roi_id, roi_keyframes, start_frame, end_frame):
    roi_keyframes = roi_keyframes.sort_values("frame").reset_index(drop=True)

    if len(roi_keyframes) == 0:
        raise ValueError(f"No keyframes found for ROI {roi_id}")

    segments = []
    first = roi_keyframes.iloc[0]

    if first["frame"] >= start_frame:
        seg_back = track_roi_segment(
            roi_id=roi_id,
            start_frame=int(first["frame"]),
            end_frame=int(start_frame),
            start_x=float(first["x_green"]),
            start_y=float(first["y_green"]),
            direction=-1
        )
        segments.append(seg_back)

    for i in range(len(roi_keyframes)):
        this_kf = roi_keyframes.iloc[i]
        this_frame = int(this_kf["frame"])

        if i < len(roi_keyframes) - 1:
            next_frame = int(roi_keyframes.iloc[i + 1]["frame"])
            segment_end = min(next_frame - 1, end_frame)
        else:
            segment_end = end_frame

        if segment_end >= this_frame:
            seg_forward = track_roi_segment(
                roi_id=roi_id,
                start_frame=this_frame,
                end_frame=segment_end,
                start_x=float(this_kf["x_green"]),
                start_y=float(this_kf["y_green"]),
                direction=+1
            )
            segments.append(seg_forward)

    roi_trace = pd.concat(segments, ignore_index=True)
    roi_trace = (
        roi_trace
        .sort_values(["roi_id", "frame", "tracking_direction"])
        .drop_duplicates(subset=["roi_id", "frame"], keep="last")
        .sort_values("frame")
        .reset_index(drop=True)
    )

    return roi_trace

def track_all_rois_with_keyframes(keyframes_df, start_frame, end_frame):
    all_traces = []
    roi_ids = sorted(keyframes_df["roi_id"].unique())

    for roi_id in tqdm(roi_ids, desc="Tracking ROIs"):
        roi_keyframes = keyframes_df[keyframes_df["roi_id"] == roi_id]
        roi_trace = track_single_roi_with_keyframes(
            roi_id=roi_id,
            roi_keyframes=roi_keyframes,
            start_frame=start_frame,
            end_frame=end_frame
        )
        all_traces.append(roi_trace)

    traces_df = pd.concat(all_traces, ignore_index=True)
    traces_df = traces_df.sort_values(["roi_id", "frame"]).reset_index(drop=True)
    return traces_df

In [ ]:
# ============================================================
# CELL 11: RUN TRACKING
# ============================================================

traces_df = track_all_rois_with_keyframes(
    keyframes_df=keyframes_df,
    start_frame=START_FRAME,
    end_frame=END_FRAME
)

raw_output_path = os.path.join(OUTPUT_DIR, "green_tracked_roi_raw_traces.csv")
traces_df.to_csv(raw_output_path, index=False)

print("Tracking complete.")
print("Saved:", raw_output_path)
display(traces_df.head())

In [ ]:
# ============================================================
# CELL 12: COMPUTE dF/F
# ============================================================

BASELINE_MODE = "percentile"
BASELINE_PERCENTILE = 20

# To use a fixed frame range instead:
# BASELINE_MODE = "frame_range"
BASELINE_FRAME_RANGE_IMAGEJ = (1, 300)

def add_dff_columns(traces_df):
    out = traces_df.copy()
    out["green_f0"] = np.nan
    out["green_dff"] = np.nan
    out["green_dff_percent"] = np.nan

    for roi_id, sub in out.groupby("roi_id"):
        idx = sub.index
        green = sub["green_corrected"].values.astype(float)
        frames = sub["frame"].values.astype(int)

        if BASELINE_MODE == "percentile":
            f0 = np.nanpercentile(green, BASELINE_PERCENTILE)
        elif BASELINE_MODE == "frame_range":
            f_start_imagej, f_end_imagej = BASELINE_FRAME_RANGE_IMAGEJ
            f_start = f_start_imagej - 1
            f_end = f_end_imagej - 1
            use = (frames >= f_start) & (frames <= f_end)
            if np.sum(use) == 0:
                raise ValueError(f"No baseline frames found for ROI {roi_id}")
            f0 = np.nanmean(green[use])
        else:
            raise ValueError("BASELINE_MODE must be percentile or frame_range")

        if not np.isfinite(f0) or f0 == 0:
            dff = np.full_like(green, np.nan, dtype=float)
        else:
            dff = (green - f0) / f0

        out.loc[idx, "green_f0"] = f0
        out.loc[idx, "green_dff"] = dff
        out.loc[idx, "green_dff_percent"] = 100 * dff

    return out

traces_df = add_dff_columns(traces_df)

dff_output_path = os.path.join(OUTPUT_DIR, "green_tracked_roi_dff_traces.csv")
traces_df.to_csv(dff_output_path, index=False)

print("Saved:", dff_output_path)
display(traces_df.head())

In [ ]:
# ============================================================
# CELL 13: PLOT CALCIUM TRACES
# ============================================================

def plot_all_traces(traces_df, value_col="green_dff", stim_frames=None):
    roi_ids = sorted(traces_df["roi_id"].unique())
    fig, ax = plt.subplots(figsize=(11, 5))

    for roi_id in roi_ids:
        sub = traces_df[traces_df["roi_id"] == roi_id]
        ax.plot(sub["time_sec"], sub[value_col], label=f"ROI {roi_id}", linewidth=1)

    if stim_frames is not None:
        for sf in stim_frames:
            ax.axvline(sf / FPS, linestyle="--", linewidth=1)

    ax.set_xlabel("Time (s)")
    ax.set_ylabel(value_col)
    ax.set_title(f"Green-channel calcium traces: {value_col}")
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.show()

plot_all_traces(traces_df, value_col="green_corrected", stim_frames=STIM_FRAMES)
plot_all_traces(traces_df, value_col="green_dff", stim_frames=STIM_FRAMES)
plot_all_traces(traces_df, value_col="green_dff_percent", stim_frames=STIM_FRAMES)

In [ ]:
# ============================================================
# CELL 14: TRACKING QUALITY SUMMARY
# ============================================================

quality_summary = (
    traces_df
    .groupby("roi_id")
    .agg(
        n_frames=("frame", "count"),
        fraction_good=("tracking_good", "mean"),
        median_corr=("tracking_corr", "median"),
        min_corr=("tracking_corr", "min"),
        max_displacement=("tracking_displacement", "max")
    )
    .reset_index()
)

quality_path = os.path.join(OUTPUT_DIR, "green_tracking_quality_summary.csv")
quality_summary.to_csv(quality_path, index=False)

print("Tracking quality summary:")
display(quality_summary)
print("Saved:", quality_path)

def plot_tracking_quality(traces_df):
    roi_ids = sorted(traces_df["roi_id"].unique())
    fig, ax = plt.subplots(figsize=(11, 4))

    for roi_id in roi_ids:
        sub = traces_df[traces_df["roi_id"] == roi_id]
        ax.plot(sub["time_sec"], sub["tracking_corr"], label=f"ROI {roi_id}", linewidth=1)

    ax.axhline(MIN_CORRELATION_ALLOWED, linestyle="--", linewidth=1)
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Template matching correlation")
    ax.set_title("Tracking confidence")
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.show()

plot_tracking_quality(traces_df)

bad_tracking_df = traces_df[~traces_df["tracking_good"]].copy()
bad_frames_summary = (
    bad_tracking_df
    .groupby("roi_id")
    .agg(
        n_bad_frames=("frame", "count"),
        first_bad_imagej_frame=("frame_imagej", "min"),
        last_bad_imagej_frame=("frame_imagej", "max")
    )
    .reset_index()
)

bad_path = os.path.join(OUTPUT_DIR, "green_tracking_bad_frames_summary.csv")
bad_frames_summary.to_csv(bad_path, index=False)

print("Bad tracking frames summary:")
display(bad_frames_summary)
print("Saved:", bad_path)

In [ ]:
# ============================================================
# CELL 15: QC OVERLAY ON SELECTED FRAMES
# ============================================================

def get_positions_at_frame(traces_df, frame_idx):
    return traces_df[traces_df["frame"] == frame_idx].copy()

def plot_tracked_rois_on_frame(traces_df, frame_idx):
    sub = get_positions_at_frame(traces_df, frame_idx)
    if len(sub) == 0:
        print(f"No tracked positions found for frame {frame_idx}")
        return

    green = get_green_frame(frame_idx)

    fig, ax = plt.subplots(figsize=(9, 10))
    ax.imshow(normalize_for_display(green), cmap="gray")
    ax.set_title(f"Tracked ROIs on green channel\nImageJ frame {frame_idx + 1}")
    add_coordinate_grid(ax, green.shape)

    for _, row in sub.iterrows():
        roi_id = int(row["roi_id"])
        x = float(row["x_green"])
        y = float(row["y_green"])
        good = bool(row["tracking_good"])

        circle = plt.Circle((x, y), ROI_RADIUS, fill=False, linewidth=1.5)
        ax.add_patch(circle)
        ax.plot(x, y, "o", markersize=3)
        label = f"{roi_id}"
        if not good:
            label += " bad"
        ax.text(
            x + ROI_RADIUS + 2,
            y,
            label,
            fontsize=9,
            bbox=dict(facecolor="white", alpha=0.65, edgecolor="none", pad=1)
        )

    plt.tight_layout()
    plt.show()

QC_FRAMES_IMAGEJ = [574, 1475, 2057]

for f_imagej in QC_FRAMES_IMAGEJ:
    f_python = f_imagej - 1
    if START_FRAME <= f_python <= END_FRAME:
        plot_tracked_rois_on_frame(traces_df, f_python)

In [ ]:
# ============================================================
# CELL 15B: PLOT ROI CENTER TRACKS ACROSS THE WHOLE EXPERIMENT
# ============================================================
#
# This plots the x,y center of each tracked ROI across all frames.
#
# Output files:
#   1. all_roi_center_tracks_on_green.png
#   2. per_roi_center_tracks_on_green.png
#
# Interpretation:
#   - Good tracking: each ROI center track stays near the same soma and moves smoothly.
#   - Bad tracking: sudden jumps, long diagonal lines, or drift onto axon/background.
#   - Red x markers in per-ROI panels mark low-confidence frames.
# ============================================================

from pathlib import Path
from matplotlib.collections import LineCollection

TRACK_BG_FRAME_IMAGEJ = ROI_SELECTION_FRAME_IMAGEJ
TRACK_BG_FRAME = TRACK_BG_FRAME_IMAGEJ - 1

# Use 1 to plot every frame.
# If plots are too dense, try 2, 5, or 10.
TRACK_SUBSAMPLE = 1

track_output_dir = Path(OUTPUT_DIR)
track_output_dir.mkdir(parents=True, exist_ok=True)


def make_line_segments(x, y):
    """
    Convert x,y points into line segments for colored trajectory plotting.
    """
    points = np.array([x, y]).T.reshape(-1, 1, 2)
    segments = np.concatenate([points[:-1], points[1:]], axis=1)
    return segments


def plot_all_roi_tracks_on_green(traces_df, bg_frame_idx):
    """
    Plot all ROI center tracks on one green-channel background image.
    """
    green = get_green_frame(bg_frame_idx)

    fig, ax = plt.subplots(figsize=(10, 12))
    ax.imshow(normalize_for_display(green), cmap="gray")

    ax.set_title(
        "Tracked ROI centers across the whole experiment\\n"
        f"Background: ImageJ frame {bg_frame_idx + 1}"
    )

    try:
        add_coordinate_grid(ax, green.shape)
    except NameError:
        ax.set_xlabel("x coordinate in green channel")
        ax.set_ylabel("y coordinate")

    roi_ids = sorted(traces_df["roi_id"].unique())
    cmap = plt.get_cmap("tab10")

    for k, roi_id in enumerate(roi_ids):
        sub = traces_df[traces_df["roi_id"] == roi_id].copy()
        sub = sub.sort_values("frame")

        if TRACK_SUBSAMPLE > 1:
            sub = sub.iloc[::TRACK_SUBSAMPLE, :]

        x = sub["x_green"].values
        y = sub["y_green"].values

        if len(x) == 0:
            continue

        color = cmap(k % 10)

        ax.plot(
            x,
            y,
            "-",
            linewidth=1.2,
            alpha=0.85,
            color=color,
            label=f"ROI {roi_id}"
        )

        # Start and end markers.
        ax.plot(x[0], y[0], "o", color=color, markersize=7)
        ax.plot(x[-1], y[-1], "^", color=color, markersize=7)

        ax.text(
            x[0] + 4,
            y[0],
            f"ROI {roi_id} start",
            color=color,
            fontsize=8,
            bbox=dict(facecolor="black", alpha=0.45, edgecolor="none", pad=1)
        )

    ax.legend(
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        fontsize=8
    )

    plt.tight_layout()

    out_path = track_output_dir / "all_roi_center_tracks_on_green.png"
    plt.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.show()

    print("Saved:", out_path)


def plot_per_roi_tracks_on_green(traces_df, bg_frame_idx):
    """
    Plot one panel per ROI.
    Each ROI track is colored by time.
    Bad tracking frames are shown as red x markers.
    """
    green = get_green_frame(bg_frame_idx)

    roi_ids = sorted(traces_df["roi_id"].unique())
    n_rois = len(roi_ids)

    fig, axes = plt.subplots(
        n_rois,
        1,
        figsize=(8, 5 * n_rois),
        squeeze=False
    )

    for row_idx, roi_id in enumerate(roi_ids):
        ax = axes[row_idx, 0]

        sub = traces_df[traces_df["roi_id"] == roi_id].copy()
        sub = sub.sort_values("frame")

        if TRACK_SUBSAMPLE > 1:
            sub = sub.iloc[::TRACK_SUBSAMPLE, :]

        x = sub["x_green"].values
        y = sub["y_green"].values
        t = sub["time_sec"].values
        good = sub["tracking_good"].values.astype(bool)

        ax.imshow(normalize_for_display(green), cmap="gray")

        # Draw trajectory colored by time.
        if len(x) > 1:
            segments = make_line_segments(x, y)

            lc = LineCollection(
                segments,
                cmap="plasma",
                linewidth=1.5,
                alpha=0.9
            )
            lc.set_array(t[:-1])
            ax.add_collection(lc)

            cbar = plt.colorbar(lc, ax=ax, pad=0.02)
            cbar.set_label("Time (s)")

        # Start and end markers.
        if len(x) > 0:
            ax.plot(x[0], y[0], "go", markersize=8, label="start")
            ax.plot(x[-1], y[-1], "r^", markersize=8, label="end")

        # Mark bad tracking frames.
        if len(x) > 0 and np.any(~good):
            ax.scatter(
                x[~good],
                y[~good],
                s=20,
                marker="x",
                color="red",
                label="bad tracking frame",
                zorder=5
            )

        dx_range = np.ptp(x) if len(x) else np.nan
        dy_range = np.ptp(y) if len(y) else np.nan

        ax.set_title(
            f"ROI {roi_id}: center track across experiment\\n"
            f"Δx = {dx_range:.1f} px, Δy = {dy_range:.1f} px"
        )

        try:
            add_coordinate_grid(ax, green.shape)
        except NameError:
            ax.set_xlabel("x coordinate in green channel")
            ax.set_ylabel("y coordinate")

        ax.legend(fontsize=8)

    plt.tight_layout()

    out_path = track_output_dir / "per_roi_center_tracks_on_green.png"
    plt.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.show()

    print("Saved:", out_path)


plot_all_roi_tracks_on_green(traces_df, TRACK_BG_FRAME)
plot_per_roi_tracks_on_green(traces_df, TRACK_BG_FRAME)

In [ ]:
# ============================================================
# CELL 15C: INTERACTIVE TRACKING MOVIE WITH PLAY/PAUSE
# ============================================================
#
# Purpose:
#   Make an interactive movie with play/pause controls that overlays
#   tracked ROI positions on the green-channel image.
#
# Output:
#   - Displays an interactive movie inside the notebook
#   - Optional: saves an mp4 if SAVE_MP4 = True
# ============================================================

import os
import sys
import subprocess
from pathlib import Path

import matplotlib as mpl
from matplotlib.animation import FuncAnimation, FFMpegWriter
from IPython.display import HTML, display

# Increase embed limit so the interactive movie can display in the notebook.
# If the movie is large and fails to display, increase this further or increase MOVIE_FRAME_STEP.
mpl.rcParams["animation.embed_limit"] = 300  # MB


# ---------------- USER SETTINGS ----------------
MOVIE_FRAME_STEP = 10     # show every 50th frame; try 20 or 10 for smoother movie
MOVIE_FPS = 5             # playback speed
MOVIE_DPI = 150

MOVIE_START_FRAME_IMAGEJ = 1
MOVIE_END_FRAME_IMAGEJ = n_frames

CROP_TO_TRACK_REGION = True
CROP_MARGIN = 30

SHOW_TRAIL = True
TRAIL_LENGTH_FRAMES = 200

SAVE_MP4 = False
MP4_FILENAME = f"roi_tracking_movie_every_{MOVIE_FRAME_STEP}_frames.mp4"
# ------------------------------------------------


movie_output_dir = Path(OUTPUT_DIR)
movie_output_dir.mkdir(parents=True, exist_ok=True)

# Convert ImageJ frame numbers to Python indexing.
movie_start = max(0, MOVIE_START_FRAME_IMAGEJ - 1)
movie_end = min(n_frames - 1, MOVIE_END_FRAME_IMAGEJ - 1)

frame_list = list(range(movie_start, movie_end + 1, MOVIE_FRAME_STEP))

if len(frame_list) == 0:
    raise ValueError("No frames selected for movie. Check frame range settings.")

# Green-channel dimensions.
green_height, green_width = get_green_frame(movie_start).shape

roi_ids = sorted(traces_df["roi_id"].unique())
cmap = plt.get_cmap("tab10")

# Precompute crop bounds from all tracked centers.
if CROP_TO_TRACK_REGION:
    x_all = traces_df["x_green"].values
    y_all = traces_df["y_green"].values

    x_min = max(0, int(np.floor(np.nanmin(x_all) - CROP_MARGIN)))
    x_max = min(green_width - 1, int(np.ceil(np.nanmax(x_all) + CROP_MARGIN)))

    y_min = max(0, int(np.floor(np.nanmin(y_all) - CROP_MARGIN)))
    y_max = min(green_height - 1, int(np.ceil(np.nanmax(y_all) + CROP_MARGIN)))
else:
    x_min, x_max = 0, green_width - 1
    y_min, y_max = 0, green_height - 1

print("Interactive movie settings:")
print(f"  Number of displayed frames: {len(frame_list)}")
print(f"  Frame step: every {MOVIE_FRAME_STEP} frames")
print(f"  Crop x: {x_min} to {x_max}")
print(f"  Crop y: {y_min} to {y_max}")


def get_display_green_crop(frame_idx):
    """
    Get normalized cropped green-channel image for display.
    """
    green = get_green_frame(frame_idx)
    green_crop = green[y_min:y_max + 1, x_min:x_max + 1]
    return normalize_for_display(green_crop)


# Initial image
first_frame = frame_list[0]
first_img = get_display_green_crop(first_frame)

fig, ax = plt.subplots(figsize=(7, 7))
im = ax.imshow(first_img, cmap="gray")

ax.set_xlim(0, first_img.shape[1] - 1)
ax.set_ylim(first_img.shape[0] - 1, 0)
ax.set_xlabel("x coordinate, cropped view")
ax.set_ylabel("y coordinate")

title = ax.set_title("")

# Artists for each ROI
circle_artists = {}
center_artists = {}
text_artists = {}
trail_artists = {}
bad_artists = {}

for k, roi_id in enumerate(roi_ids):
    color = cmap(k % 10)

    circ = plt.Circle(
        (0, 0),
        ROI_RADIUS,
        fill=False,
        lw=1.8,
        color=color,
        visible=False
    )
    ax.add_patch(circ)

    center_dot, = ax.plot(
        [],
        [],
        "o",
        color=color,
        ms=4,
        visible=False
    )

    label_text = ax.text(
        0,
        0,
        f"ROI {roi_id}",
        color=color,
        fontsize=8,
        visible=False,
        bbox=dict(
            facecolor="black",
            alpha=0.5,
            edgecolor="none",
            pad=1
        )
    )

    trail_line, = ax.plot(
        [],
        [],
        "-",
        lw=1.0,
        color=color,
        alpha=0.75,
        visible=False
    )

    bad_marker, = ax.plot(
        [],
        [],
        "x",
        color="red",
        ms=8,
        mew=2,
        visible=False
    )

    circle_artists[roi_id] = circ
    center_artists[roi_id] = center_dot
    text_artists[roi_id] = label_text
    trail_artists[roi_id] = trail_line
    bad_artists[roi_id] = bad_marker


def update_movie(frame_i):
    """
    Update function for interactive animation.
    """
    frame_idx = frame_list[frame_i]

    # Update background image.
    im.set_data(get_display_green_crop(frame_idx))

    # Current ROI positions at this frame.
    sub_now = traces_df[traces_df["frame"] == frame_idx].copy()

    time_sec = frame_idx / FPS

    title.set_text(
        f"Tracked ROIs on green channel | ImageJ frame {frame_idx + 1} / {n_frames}\n"
        f"Time = {time_sec:.1f} s | displaying every {MOVIE_FRAME_STEP} frames"
    )

    for roi_id in roi_ids:
        circ = circle_artists[roi_id]
        center_dot = center_artists[roi_id]
        label_text = text_artists[roi_id]
        trail_line = trail_artists[roi_id]
        bad_marker = bad_artists[roi_id]

        sub_roi = sub_now[sub_now["roi_id"] == roi_id]

        if len(sub_roi) == 0:
            circ.set_visible(False)
            center_dot.set_visible(False)
            label_text.set_visible(False)
            trail_line.set_visible(False)
            bad_marker.set_visible(False)
            continue

        x = float(sub_roi["x_green"].iloc[0])
        y = float(sub_roi["y_green"].iloc[0])
        good = bool(sub_roi["tracking_good"].iloc[0])

        # Convert to cropped coordinates.
        xc = x - x_min
        yc = y - y_min

        # Current ROI circle.
        circ.center = (xc, yc)
        circ.set_visible(True)

        # Current center dot.
        center_dot.set_data([xc], [yc])
        center_dot.set_visible(True)

        # Label.
        label_text.set_position((xc + ROI_RADIUS + 2, yc))
        label_text.set_text(f"ROI {roi_id}")
        label_text.set_visible(True)

        # Bad tracking marker.
        if good:
            bad_marker.set_visible(False)
        else:
            bad_marker.set_data([xc], [yc])
            bad_marker.set_visible(True)

        # Trail.
        if SHOW_TRAIL:
            sub_hist = traces_df[
                (traces_df["roi_id"] == roi_id) &
                (traces_df["frame"] <= frame_idx) &
                (traces_df["frame"] >= frame_idx - TRAIL_LENGTH_FRAMES)
            ].copy().sort_values("frame")

            if len(sub_hist) >= 2:
                xh = sub_hist["x_green"].values - x_min
                yh = sub_hist["y_green"].values - y_min

                trail_line.set_data(xh, yh)
                trail_line.set_visible(True)
            else:
                trail_line.set_visible(False)
        else:
            trail_line.set_visible(False)

    artists = [im, title]
    artists += list(circle_artists.values())
    artists += list(center_artists.values())
    artists += list(text_artists.values())
    artists += list(trail_artists.values())
    artists += list(bad_artists.values())

    return artists


anim_roi_tracking = FuncAnimation(
    fig,
    update_movie,
    frames=len(frame_list),
    interval=1000 / MOVIE_FPS,
    blit=False,
    cache_frame_data=False
)

# Optional mp4 saving.
if SAVE_MP4:
    mp4_path = movie_output_dir / MP4_FILENAME
    writer = FFMpegWriter(fps=MOVIE_FPS)
    anim_roi_tracking.save(mp4_path, writer=writer, dpi=MOVIE_DPI)
    print("Saved mp4:", mp4_path)

plt.close(fig)

display(HTML(anim_roi_tracking.to_jshtml()))

print("Interactive movie object created: anim_roi_tracking")

In [ ]:
# ============================================================
# CELL 16: MANUAL CORRECTION HELPER
# ============================================================
#
# If tracking fails:
# 1. Change FRAME_TO_INSPECT_IMAGEJ below.
# 2. Read the corrected x,y coordinate using the grid.
# 3. Add it to MANUAL_KEYFRAMES in CELL 9.
# 4. Rerun CELL 9 onward.
# ============================================================

FRAME_TO_INSPECT_IMAGEJ = 1200

if 1 <= FRAME_TO_INSPECT_IMAGEJ <= n_frames:
    show_green_frame_big(
        FRAME_TO_INSPECT_IMAGEJ - 1,
        title_extra="Use this grid to read x,y coordinates for manual correction",
        figsize=(9, 10)
    )
else:
    print("FRAME_TO_INSPECT_IMAGEJ is outside the analyzed movie range.")

In [ ]:
# ============================================================
# CELL 17: SAVE WIDE-FORM TRACE TABLES
# ============================================================

def save_wide_trace_tables(traces_df):
    value_cols = [
        "green_corrected",
        "green_dff",
        "green_dff_percent",
        "tracking_corr",
        "tracking_good",
        "x_green",
        "y_green"
    ]

    for value_col in value_cols:
        wide = traces_df.pivot(index="frame_imagej", columns="roi_id", values=value_col)
        wide.columns = [f"ROI_{int(c)}" for c in wide.columns]
        wide = wide.reset_index()

        out_path = os.path.join(OUTPUT_DIR, f"wide_{value_col}.csv")
        wide.to_csv(out_path, index=False)
        print("Saved:", out_path)

save_wide_trace_tables(traces_df)